In [1]:
import pandas as pd
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
A=pd.read_excel("/content/A.xlsx")
B=pd.read_excel("/content/B.xlsx")
C=pd.read_excel("/content/C.xlsx")
D=pd.read_excel("/content/D.xlsx")

In [3]:
# ============================================================
# Federated Learning (DeepMLP) + αᵢ Mitigation
# + SMOTE + FocalLoss + FedProx + FedBN-style Aggregation + Fine-tuning
# ============================================================

import os, math, copy, random, argparse
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE

# ------------------------------------------------
# 0) 결함 라벨 정의
# ------------------------------------------------
DEFECT_LABELS = ['Short_Shot_1', 'Bubble_1', 'Exfoliation_1', 'Blow_Hole_1', 'Deformation_1']

# ------------------------------------------------
# 1) 데이터 로드 + SMOTE Oversampling
# ------------------------------------------------
def oversample_dataset(X, y):
    sm = SMOTE(random_state=42, k_neighbors=5)
    X_res, y_res = sm.fit_resample(X, y)
    return X_res, y_res

def load_excel_binary(path: str):
    df = pd.read_excel(path).dropna(axis=1, how='all')
    for col in DEFECT_LABELS:
        if col not in df.columns:
            df[col] = 0

    y = df[DEFECT_LABELS].sum(axis=1).clip(upper=1).astype(int).values
    X = (
        df.drop(columns=DEFECT_LABELS, errors='ignore')
          .select_dtypes(include=[np.number])
          .replace([np.inf, -np.inf], np.nan)
          .dropna(axis=0, how='any')
    )
    y = y[:len(X)]
    X_res, y_res = oversample_dataset(X, y)
    return X_res, y_res

def build_clients(path_map):
    raw = {}
    for cid, path in path_map.items():
        X, y = load_excel_binary(path)
        raw[cid] = {"X": X, "y": y}
        print(f"[Client {cid}] after SMOTE → shape: {X.shape}, class balance: {pd.Series(y).value_counts().to_dict()}")

    cols = set.intersection(*[set(v["X"].columns) for v in raw.values()])
    clients, metas = {}, {}
    for cid, v in raw.items():
        feat = v["X"][list(cols)].copy()
        mu, sigma = feat.mean(), feat.std().replace(0, 1)
        feat = (feat - mu) / sigma
        X = torch.tensor(feat.values.astype(np.float32))
        y = torch.tensor(v["y"], dtype=torch.long)
        allset = TensorDataset(X, y)
        n = len(allset)
        n_train = int(0.8 * n)
        n_test = n - n_train
        tr, te = random_split(allset, [n_train, n_test], generator=torch.Generator().manual_seed(42))
        clients[cid] = {"train": tr, "test": te, "all": allset}
        metas[cid] = {
            "n_samples": len(y),
            "class_counts": pd.Series(v["y"]).value_counts().to_dict()
        }
    return clients, metas, list(cols)

# ------------------------------------------------
# 2) DeepMLP 모델 정의
# ------------------------------------------------
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)

# ------------------------------------------------
# 3) Focal Loss (불균형 대응)
# ------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(self, alpha=2.0, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        pt = torch.exp(-ce_loss)
        return (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()

# ------------------------------------------------
# 4) αᵢ 및 FedBN Aggregation
# ------------------------------------------------
def compute_ci(class_counts, eps_frac=0.05, min_samples=1):
    K = len(class_counts)
    total = sum(class_counts.values())
    if K == 0 or total == 0:
        return 1e-8
    H = -sum((c / total) * math.log(c / total) for c in class_counts.values() if c > 0)
    logK = math.log(K + 1e-12)
    H_floor = eps_frac * logK
    H_tilde = max(H, H_floor)
    k_i = sum(1 for c in class_counts.values() if c >= min_samples)
    frac_k = k_i / K
    return max(1e-8, frac_k * (H_tilde / logK))

def aggregate_with_softmax_weights(state_dicts, alpha_components, rho=1.0, beta=1.0, theta=0.8, use_fedbn=True):
    if not state_dicts:
        return None
    logits = {}
    for cid, comp in alpha_components.items():
        logits[cid] = (
            math.log(comp["n_i"] ** rho + 1e-12)
            + math.log(comp["c_i"] ** beta + 1e-12)
            + math.log(comp["t_i"] ** theta + 1e-12)
        )
    max_logit = max(logits.values())
    exp_logits = {cid: math.exp(logits[cid] - max_logit) for cid in logits}
    total_exp = sum(exp_logits.values())
    alphas = {cid: exp_logits[cid] / total_exp for cid in logits}

    base_sd = next(iter(state_dicts.values()))
    global_sd = {}
    for k in base_sd.keys():
        if use_fedbn and ("bn" in k or "running_mean" in k or "running_var" in k):
            global_sd[k] = base_sd[k].clone()
        else:
            agg = sum(sd[k] * alphas[cid] for cid, sd in state_dicts.items())
            global_sd[k] = agg.clone()
    return global_sd

# ------------------------------------------------
# 5) Local Train (FedProx 적용)
# ------------------------------------------------
def local_train(base_model, dataset, lr, epochs, device, global_params=None, mu=0.0):
    model = copy.deepcopy(base_model).to(device)
    model.train()
    crit = FocalLoss(alpha=2.0, gamma=1.5)
    opt = optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(dataset, batch_size=64, shuffle=True)

    global_params = [p.detach().clone() for p in base_model.parameters()] if mu > 0 else None
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = crit(logits, y)
            if global_params is not None:
                prox = sum(torch.norm(p - g) ** 2 for p, g in zip(model.parameters(), global_params))
                loss += 0.5 * mu * prox
            loss.backward()
            opt.step()
    return model.state_dict()

def evaluate(model, dataset, device, threshold=0.35, return_preds=False):
    model.eval()
    loader = DataLoader(dataset, batch_size=256)
    y_true, y_pred = [], []
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            probs = torch.softmax(model(x), dim=1)[:, 1]
            preds = (probs > threshold).long()
            y_true += y.cpu().tolist()
            y_pred += preds.cpu().tolist()
    acc = np.mean(np.array(y_true) == np.array(y_pred))
    if return_preds:
        return acc, y_true, y_pred
    return acc

def find_best_threshold(model, dataset, device, thresholds=np.arange(0.3, 0.51, 0.05)):
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        _, yt, yp = evaluate(model, dataset, device, threshold=t, return_preds=True)
        f1 = f1_score(yt, yp, average='binary', zero_division=0)
        print(f"Threshold={t:.2f}, F1={f1:.4f}")
        if f1 > best_f1:
            best_t, best_f1 = t, f1
    print(f"\n Best threshold = {best_t:.2f} (F1={best_f1:.4f})")
    return best_t

# ------------------------------------------------
# 6) Main Loop (연합학습)
# ------------------------------------------------
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--rounds", type=int, default=50)
    parser.add_argument("--mu", type=float, default=0.003)
    parser.add_argument("--ft_epochs", type=int, default=5)
    args, _ = parser.parse_known_args()

    paths = {"A": "A.xlsx", "B": "B.xlsx", "C": "C.xlsx", "D": "D.xlsx"}
    clients, metas, cols = build_clients(paths)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    input_dim = clients["A"]["all"].tensors[0].shape[1]
    global_model = DeepMLP(input_dim=input_dim, hidden_dim=128, num_classes=2).to(device)

    participation_probs = {"A": 0.9, "D": 0.7, "B": 0.4, "C": 0.4}
    combined_test = torch.utils.data.ConcatDataset([v["test"] for v in clients.values()])

    for rnd in range(1, args.rounds + 1):
        print(f"\n=== Round {rnd} ===")
        selected = [cid for cid in clients if random.random() < participation_probs[cid]]
        if len(selected) == 0:
            selected = [random.choice(list(clients.keys()))]
        state_dicts, alpha_components = {}, {}

        before_acc = evaluate(global_model, combined_test, device)
        global_params = [p.detach().clone() for p in global_model.parameters()]
        for cid in selected:
            ds = clients[cid]["train"]
            labels = [int(y) for _, y in ds]
            class_counts = pd.Series(labels).value_counts().to_dict()
            state_dict = local_train(
                base_model=global_model,
                dataset=ds,
                lr=0.0008,
                epochs=5,
                device=device,
                global_params=global_params,
                mu=args.mu
            )
            state_dicts[cid] = state_dict
            n_i = math.sqrt(metas[cid]["n_samples"])
            c_i = compute_ci(class_counts)
            t_i = participation_probs[cid]
            alpha_components[cid] = {"n_i": n_i, "c_i": c_i, "t_i": t_i}

        new_global = aggregate_with_softmax_weights(state_dicts, alpha_components, rho=1.0, beta=1.0, theta=0.8, use_fedbn=True)
        if new_global is None:
            continue
        global_model.load_state_dict(new_global)
        after_acc = evaluate(global_model, combined_test, device)
        print(f"[Round {rnd}] Global Accuracy: {before_acc:.4f} → {after_acc:.4f}")

    print("\n=== Global Evaluation ===")
    best_t = find_best_threshold(global_model, combined_test, device)
    global_acc, yt, yp = evaluate(global_model, combined_test, device, return_preds=True, threshold=best_t)
    print(f"\nFinal Global Accuracy: {global_acc:.4f}")
    print(classification_report(yt, yp, digits=4))

    print("\n=== Per-Client Fine-tuning ===")
    for cid, data in clients.items():
        print(f"\n--- Client {cid} Fine-tuning ---")
        ft_state = local_train(global_model, data["train"], lr=0.0008, epochs=args.ft_epochs, device=device)
        local_model = copy.deepcopy(global_model)
        local_model.load_state_dict(ft_state)
        acc_c, yt_c, yp_c = evaluate(local_model, data["test"], device, threshold=best_t, return_preds=True)
        print(f"[Client {cid}] Fine-tuned Accuracy: {acc_c:.4f}")
        print(classification_report(yt_c, yp_c, digits=4))


[Client A] after SMOTE → shape: (5662, 16), class balance: {0: 2831, 1: 2831}
[Client B] after SMOTE → shape: (948, 16), class balance: {0: 474, 1: 474}
[Client C] after SMOTE → shape: (5830, 16), class balance: {0: 2915, 1: 2915}
[Client D] after SMOTE → shape: (812, 16), class balance: {0: 406, 1: 406}

=== Round 1 ===
[Round 1] Global Accuracy: 0.5038 → 0.6037

=== Round 2 ===
[Round 2] Global Accuracy: 0.6037 → 0.6327

=== Round 3 ===
[Round 3] Global Accuracy: 0.6327 → 0.6297

=== Round 4 ===
[Round 4] Global Accuracy: 0.6297 → 0.6282

=== Round 5 ===
[Round 5] Global Accuracy: 0.6282 → 0.6606

=== Round 6 ===
[Round 6] Global Accuracy: 0.6606 → 0.6595

=== Round 7 ===
[Round 7] Global Accuracy: 0.6595 → 0.6640

=== Round 8 ===
[Round 8] Global Accuracy: 0.6640 → 0.6505

=== Round 9 ===
[Round 9] Global Accuracy: 0.6505 → 0.6723

=== Round 10 ===
[Round 10] Global Accuracy: 0.6723 → 0.6727

=== Round 11 ===
[Round 11] Global Accuracy: 0.6727 → 0.6761

=== Round 12 ===
[Round 12] G